# Calibración química y CO₂: partición principal y prueba inversa
**Principal: calibrar CO₂ con LAB019–021; holdout CO₂ LAB016–018.**
**Sensibilidad inversa: calibrar CO₂ con LAB016–018; holdout CO₂ LAB019–021.**

Las tres formulaciones reajustan únicamente sus coeficientes de CO₂ en cada partición, con la misma señal cruda reducida a medianas horarias, cobertura y ponderación por fermentación. La química sigue calibrada con LAB004–009. Sus parámetros guardados se reutilizan por defecto; activar REFIT_CHEMISTRY sólo para repetir ese ajuste.

El histórico conserva sus ecuaciones y theta_nv2: se reajustan kCO2_release_h, CO2sat_scale y matrix_gain para esta comparación. Los ajustes históricos anteriores se conservan en sus archivos originales; no se sustituyen. Los modelos nuevos ajustan kLa y gain. Misma inicialización por modelo en ambas particiones; ningún dato de holdout entra al objetivo.

Solubilidad de los modelos nuevos PROVISIONAL; Cd0 de LAB021 provisional y Cd0 de LAB016–018 asumido cero. Los resultados son ajustes locales y no validan estos supuestos. Sólo se guardan parámetros en configs/models, sin CSV ni PNG adicionales.

In [ ]:
from pathlib import Path
import sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
assert Path(sys.prefix).name.lower()=='fermentation'


In [ ]:
from wine_fermentation.data import natural_batches
from wine_fermentation.shared.optimization import fit
REFIT_CHEMISTRY=False
fits={}
if REFIT_CHEMISTRY:
    _,batches,_=natural_batches(ROOT)
for name in ('without_transporters','with_transporters'):
    fits[name]=fit(ROOT,name,batches) if REFIT_CHEMISTRY else json.loads((ROOT/f'configs/models/{name}/fit.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{k:v for k,v in f.items() if k in ('model','success','initial_objective','final_objective','calls','seconds','message')} for f in fits.values()]))


In [ ]:
from wine_fermentation.comparison import run_both,PRIMARY_TRAINING
from IPython.display import Markdown
experiments=run_both(ROOT,fit_gas=True)
for key,res in experiments.items():
    display(Markdown('## '+('Principal' if key==PRIMARY_TRAINING else 'Inversa')+': '+key))
    display(pd.DataFrame(res['gas_fits']).T)
    display(res['metrics'])
summary=pd.concat([res['metrics'] for res in experiments.values()],ignore_index=True)
display(Markdown('## Comparación de CO₂: media de RMSE por fermentación [g/(L·h)]'))
display(summary.groupby(['model','training','split']).RMSE.mean().unstack(['training','split']))
